In [1]:
import requests, json

OPENAI_API_KEY = "EMPTY"
with open("../.openai_key.txt") as f:
    OPENAI_API_KEY = f.read().strip()
API_URL = "https://api.openai.com/v1/chat/completions"
MODEL = "gpt-3.5-turbo"  # Change to "gpt-4" if needed

HEADERS = {
    "Authorization": f"Bearer {OPENAI_API_KEY}",
    "Content-Type": "application/json"
}

def _send_request(messages, api=API_URL, headers=HEADERS, model=MODEL):
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.0 
    }

    response = requests.post(api, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    reply = data["choices"][0]["message"]["content"]
    return reply

In [2]:
prompt = "You need to classify a given text as anonymized or not anonymized. Just respond with you classification. This is the text: "

def build_messages(prompt_appendix, text_):
    return [{"role": "user", "content": prompt_appendix + text_}]

In [3]:
from utils.read_jsonl import read_jsonl

eval_df = read_jsonl("../DB-bio/combined_val_and_val_sft_anonymized.jsonl")

In [4]:
eval_df = eval_df[eval_df["text"].apply(len) < 1100]
len(eval_df)

131

In [11]:
def get_llm_classification(row):
    print(row.name, row.label)
    true_label = row["label"]
    _text = row["text"]
    explanation = _send_request(build_messages(prompt, _text))
    print(explanation)
    return explanation

In [12]:
len(eval_df)

131

In [13]:
test_df = eval_df.head(3).copy()

In [15]:
eval_df["explanation"] = eval_df.apply(get_llm_classification,axis=1)

2 0
Not anonymized
8 0
Not anonymized
11 0
Not anonymized
12 0
Not anonymized
14 0
Not anonymized
18 0
Not anonymized
19 0
Not anonymized
29 0
Not anonymized
32 0
Not anonymized
33 0
Not anonymized
34 0
Not anonymized
45 0
Not anonymized
49 0
Not anonymized
53 0
Not anonymized
55 0
Not anonymized
56 0
Not anonymized
57 0
Not anonymized
60 0
Not anonymized
62 0
Not anonymized
63 0
Not anonymized
70 0
Not anonymized
73 0
Not anonymized
75 0
Not anonymized
76 0
Not anonymized
83 0
Not anonymized
86 0
Not anonymized
87 0
Not anonymized
88 0
Not anonymized
91 0
Not anonymized
92 0
Not anonymized
95 0
Not anonymized
98 0
Not anonymized
100 0
Not anonymized
106 0
Not anonymized
115 0
Not anonymized
117 0
Not anonymized
122 0
Not anonymized
128 0
Not anonymized
132 0
Not anonymized
135 0
Not anonymized
137 0
Not anonymized
139 0
Not anonymized
140 0
Not anonymized
143 0
Not anonymized
146 0
Not anonymized
154 0
Not anonymized
156 0
Not anonymized
157 0
Not anonymized
161 0
Not anonymized
174 0

In [12]:
eval_df.to_csv("../4_ExplanationResults/llm_classification.csv", index=False)

NameError: name 'eval_df' is not defined

In [10]:
eval_df.head(1)["explanation"]

KeyError: 'explanation'

In [42]:
eval_df.tail(1)["explanation"]

485    This text is classified as anonymized because ...
Name: explanation, dtype: object

In [1]:
with open("../4_ExplanationResults/LLM_classsification_results.txt", "r") as f:
    lines = f.readlines()


In [4]:
i = 0
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0

while i < len(lines):
    true_label = lines[i].split(" ")[1]
    llm_response = lines[i+1].strip()
    if int(true_label) == 1:
        if llm_response == "Anonymized":
            true_positives += 1
        elif llm_response == "Not anonymized":
            false_negatives +=1
        else:
            print(llm_response)
    elif int(true_label) == 0:
        if llm_response == "Anonymized":
            false_positives += 1
        elif llm_response == "Not anonymized":
            true_negatives += 1
        else:
            print(llm_response)
    i += 2

In [7]:
true_positives ,true_negatives,false_positives ,false_negatives

(10, 71, 0, 50)

In [10]:
def compute_metrics(tp, tn, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0

    return {
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "Specificity": specificity
    }

In [11]:
compute_metrics(true_positives, true_negatives, false_positives, false_negatives)

{'Precision': 1.0,
 'Recall': 0.16666666666666666,
 'F1 Score': 0.2857142857142857,
 'Accuracy': 0.6183206106870229,
 'Specificity': 1.0}